# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 03

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В датафрейме столбец `registered_at` хранит даты регистрации клиентов (например, `'2025-05-12'`), но имеет тип данных `object` (текстовая строка).
Сможет ли аналитик напрямую вычислить разницу в днях между датами (например, `df['registered_at'] - datetime.now()`)? Какая функция Pandas преобразует столбец в специальный тип даты/времени?

**Ответ:**
Нет, со строками типа `object` математические операции дат вызовут ошибку `TypeError`. Для перевода строк в дату/время используется функция `pd.to_datetime(df['registered_at'])`.

---

#### 1.2. Аналитик применил метод `df['balance'].describe()` к столбцу баланса клиентов и получил:
`mean = 1450.0`, `50% (медиана) = 500.0`, `max = 7500.0`.
О чем говорит такой сильный разрыв между средним арифметическим (1450) и медианой (500)? Почему медиана лучше отражает типичный баланс клиента клуба?

**Ответ:**
Разрыв говорит о наличии небольшого числа клиентов с очень крупным балансом (выбросы/VIP-клиенты), которые сильно завышают среднее арифметическое. Медиана (500 руб.) показывает, что ровно у половины клиентов баланс не превышает 500 руб., поэтому она лучше характеризует типичного посетителя.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["is_long_session"] = df["duration_hours"] > df["duration_hours"].median()
```

**Ответ:**
Формирует бинарный (булев) признак: превышает ли продолжительность сессии медианное значение длительности сессий клуба (удобно для разбивки на короткие и длинные сессии).

---

#### 1.4. Почему следующий код фильтрации в Pandas упадет с ошибкой или сработает неверно?
```python
# Ошибочный код:
df[df['balance'] > 1000 & df['city'] == 'Москва']
```
Как правильно записать это условие со скобками и почему круглые скобки обязательны?

**Ответ:**
В Python побитовый оператор `&` имеет более высокий приоритет, чем операции сравнения `>` и `==`. Без скобок интерпретатор пытается вычислить `1000 & df['city']`. Правильная запись: `df[(df['balance'] > 1000) & (df['city'] == 'Москва')]`.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `hardware_pcs.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("hardware_pcs.csv").parent if os.path.exists("hardware_pcs.csv") else pathlib.Path("../variant_03")
df_audit = pd.read_csv(data_dir / "hardware_pcs.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Средняя часовая ставка ПК: 232.00")

Количество строк в таблице: 30
Столбцы датафрейма: ['pc_id', 'pc_number', 'zone_name', 'hourly_rate', 'gpu', 'cpu', 'ram_gb', 'status']
Средняя часовая ставка ПК: 232.00

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем полей является точка с запятой (`;`). Без параметра `sep=';'` столбцы склеиваются в один.
2. Опечатка в названии метода сброса индекса: вызван несуществующий метод `.rest_index()` вместо `.reset_index()`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_03")
df_bar = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

cat_summary = df_bar.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

cat_summary["quantity"] = cat_summary["quantity"].round(2)
cat_summary = cat_summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(cat_summary)

,category,total_amount,quantity
0,Мерч,108600,1.54
1,Напитки,28880,1.52
2,Горячее,25790,1.48
3,Снеки,19990,1.42


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `hardware_pcs` столбец `pc_number` (например, 'PC-01') имеет ограничение `UNIQUE`.
От какой проблемы учета это защищает клуб, и может ли столбец с ограничением `UNIQUE` содержать значение `NULL`?

**Ответ:**
Ограничение `UNIQUE` гарантирует, что в клубе не появятся два компьютера с одинаковым инвентарным номером 'PC-01'. В отличие от `PRIMARY KEY`, столбец с ограничением `UNIQUE` в SQL может содержать значение `NULL` (отсутствие номера).

---

#### 3.2. Разработчик через скрипт на Python выполнил добавление новых компьютеров командой `cursor.execute("INSERT INTO hardware_pcs ...")`, но после закрытия программы обнаружил, что компьютеры не сохранились в файле базы данных.
Какую обязательную команду для объекта соединения (`conn`) он забыл вызвать и почему?

**Ответ:**
Он забыл вызвать команду фиксации транзакции `conn.commit()`. В библиотеке `sqlite3` изменения, внесенные через курсор, по умолчанию находятся во временной транзакции и не сохраняются на диске без явного `commit()`.

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT 
    zone_name,
    COUNT(*) AS pcs_count,
    ROUND(AVG(hourly_rate), 2) AS avg_rate
FROM hardware_pcs
GROUP BY zone_name
ORDER BY avg_rate DESC;
```

**Ответ:**
Запрос группирует парк компьютеров клуба по зонам (`zone_name`) и вычисляет: общее число ПК в каждой зоне (`COUNT(*)`) и среднюю почасовую ставку аренды (`AVG(hourly_rate)`). Результаты упорядочены по убыванию средней ставки.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_03")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT member_id, nickname, city, loyalty_tier, balance, registered_at
FROM members
WHERE city = 'Москва' AND loyalty_tier IN ('Gold', 'Platinum')
ORDER BY balance DESC, member_id ASC
LIMIT 6;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,member_id,nickname,city,loyalty_tier,balance,registered_at
0,131,Ranger_543,Москва,Gold,7500.0,2025-09-05
1,41,Bullet_645,Москва,Gold,4000.0,2025-04-07
2,73,Blade_298,Москва,Gold,4000.0,2025-10-31
3,84,Tempest_501,Москва,Gold,4000.0,2025-11-20
4,100,Specter_899,Москва,Platinum,4000.0,2025-04-21
5,144,Ranger_894,Москва,Gold,4000.0,2025-01-24


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT 
    city,
    COUNT(*) AS members_count,
    ROUND(SUM(balance), 2) AS total_balance,
    ROUND(AVG(balance), 2) AS avg_balance
FROM members
GROUP BY city
HAVING COUNT(*) >= 15
ORDER BY members_count DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,city,members_count,total_balance,avg_balance
0,Москва,64,116600.0,1821.88
1,Санкт-Петербург,43,106150.0,2468.60
2,Казань,18,18950.0,1052.78
3,Екатеринбург,15,11750.0,783.33


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT 
    p.pc_id,
    p.pc_number,
    p.zone_name,
    p.hourly_rate,
    COUNT(s.session_id) AS sessions_count,
    ROUND(SUM(s.total_paid), 2) AS total_pc_revenue
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.pc_id, p.pc_number, p.zone_name, p.hourly_rate
ORDER BY total_pc_revenue DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,pc_id,pc_number,zone_name,hourly_rate,sessions_count,total_pc_revenue
0,26,PC-26,VIP,400.0,19,19200.0
1,24,PC-24,VIP,400.0,23,18380.0
2,23,PC-23,VIP,400.0,20,16680.0
3,25,PC-25,VIP,380.0,15,12882.0
4,15,PC-15,Bootcamp,270.0,17,12339.0
5,19,PC-19,Bootcamp,250.0,23,12063.0
6,20,PC-20,Bootcamp,250.0,21,11100.0
7,16,PC-16,Bootcamp,250.0,21,10826.0
8,21,PC-21,Bootcamp,270.0,22,10759.0
9,22,PC-22,Bootcamp,250.0,21,10563.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE balance > (SELECT AVG(balance) FROM members)
ORDER BY balance DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,member_id,nickname,city,loyalty_tier,balance
0,10,Cobra_470,Новосибирск,Bronze,7500.0
1,22,Lynx_155,Санкт-Петербург,Silver,7500.0
2,54,Dragon_907,Санкт-Петербург,Bronze,7500.0
3,59,Specter_367,Москва,Bronze,7500.0
4,68,Thunder_241,Санкт-Петербург,Gold,7500.0
5,70,Specter_960,Санкт-Петербург,Bronze,7500.0
6,79,Specter_390,Санкт-Петербург,Gold,7500.0
7,91,Ronin_620,Санкт-Петербург,Platinum,7500.0
8,96,Vortex_522,Новосибирск,Silver,7500.0
9,115,Ghost_907,Санкт-Петербург,Bronze,7500.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Отбор активных компьютеров

**Найденные ошибки:**
1. Синтаксическая ошибка в фильтрации: условия в блоке `WHERE` разделены запятой вместо логического оператора `AND`.

In [9]:
query_51_fixed = '''
SELECT 
    pc_number,
    zone_name,
    hourly_rate,
    status
FROM hardware_pcs
WHERE status = 'Active' AND hourly_rate > 200
ORDER BY hourly_rate DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,pc_number,zone_name,hourly_rate,status
0,PC-23,VIP,400.0,Active
1,PC-24,VIP,400.0,Active
2,PC-26,VIP,400.0,Active
3,PC-25,VIP,380.0,Active
4,PC-15,Bootcamp,270.0,Active
5,PC-21,Bootcamp,270.0,Active
6,PC-16,Bootcamp,250.0,Active
7,PC-17,Bootcamp,250.0,Active
8,PC-18,Bootcamp,250.0,Active
9,PC-19,Bootcamp,250.0,Active


#### 5.2. Средняя стоимость аренды компьютеров по зонам

**Найденные ошибки:**
1. Синтаксический порядок: секция `WHERE` написана после `GROUP BY`, фильтрация групп должна выполняться через `HAVING`.
2. Использование псевдонима: в фильтрации указан alias `avg_rate` вместо агрегатного выражения `AVG(hourly_rate)`.

In [10]:
query_52_fixed = '''
SELECT 
    zone_name,
    COUNT(*) AS pcs_count,
    ROUND(AVG(hourly_rate), 2) AS avg_rate
FROM hardware_pcs
GROUP BY zone_name
HAVING AVG(hourly_rate) > 200
ORDER BY avg_rate DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,zone_name,pcs_count,avg_rate
0,VIP,5,396.00
1,PS5,3,306.67
2,Bootcamp,8,255.00


#### 5.3. Сравнение расходов в баре со средним чеком бара

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = b.order_id` связывает клиента с номером заказа вместо `b.member_id`.
2. Неполный `GROUP BY`: поле `m.nickname` выбрано в `SELECT`, но отсутствует в списке группировки.
3. Многострочный подзапрос: группировка `GROUP BY category` в подзапросе возвращает набор строк вместо одного скалярного среднего.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    ROUND(AVG(b.total_amount), 2) AS avg_order_sum
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
GROUP BY m.member_id, m.nickname
HAVING AVG(b.total_amount) > (
    SELECT AVG(total_amount) 
    FROM bar_orders
)
ORDER BY avg_order_sum DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,avg_order_sum
0,102,Warlock_563,2590.00
1,130,Sentinel_316,2363.33
2,79,Specter_390,1907.50
3,133,Wolf_650,1624.00
4,78,Kaiser_603,1600.00
5,47,Tempest_560,1596.67
6,59,Specter_367,1595.00
7,55,Neo_908,1143.33
8,68,Thunder_241,1101.43
9,101,CyberKnight_448,1095.00
